# WikiHow-MY — NLLB-200-distilled-600M fine-tune (en→my) on Colab

Heavy GPU training only. Everything else (data, eval, paper) runs locally via Claude Code.

**Before running:** upload the project folder (with `data/processed/`, `src/`, `scripts/`) to your Google Drive, e.g. `MyDrive/wikihow-my/`. Set `PROJECT` below. Runtime → Change runtime type → **GPU (T4 ok; A100 faster)**.

Pipeline: fine-tune → inference (zero-shot + fine-tuned) → score chrF++/spBLEU/BLEU + IFS → COMET (installed *after* training) → results table. All numbers land in `experiments/results/main_results.json`.

In [ ]:
# Training/eval deps only. unbabel-comet is installed LATER (after training) so its
# dependency pins can't disturb the training stack. (myanmar-tools removed: not on PyPI;
# the Zawgyi/Unicode check was done at data-prep time.)
!pip -q install -U transformers datasets sacrebleu sentencepiece pyyaml accelerate

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT = '/content/drive/MyDrive/wikihow-my'  # <-- edit to your path
os.chdir(PROJECT)
print('cwd:', os.getcwd())
assert os.path.exists('data/processed/train.jsonl'), 'upload data/processed/ first'

In [ ]:
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 1. Fine-tune (early-stops on dev chrF)

In [ ]:
!python src/train/finetune_nllb.py --config src/train/config.yaml --out_dir /content/ckpt
# On T4 OOM: lower per_device_train_batch_size to 4 and gradient_accumulation_steps to 16 in config.yaml

## 2. Inference: zero-shot + fine-tuned on the test set

In [ ]:
!python src/infer/translate.py --split test --system nllb_zeroshot
!python src/infer/translate.py --split test --system nllb_finetuned --model /content/ckpt/best

## 3. Score (chrF++ / spBLEU / BLEU + IFS)

COMET is computed in §3b below (after installing `unbabel-comet`), so its dependencies cannot disturb training.

In [ ]:
# chrF++ / spBLEU / BLEU (reference-based) + IFS (source-anchored procedural faithfulness).
!python src/eval/automatic.py --hyps experiments/results/nllb_zeroshot_test_hyps.txt --refs data/processed/test.jsonl --system nllb_zeroshot
!python src/eval/automatic.py --hyps experiments/results/nllb_finetuned_test_hyps.txt --refs data/processed/test.jsonl --system nllb_finetuned
!python src/eval/ifs.py --hyps experiments/results/nllb_zeroshot_test_hyps.txt --refs data/processed/test.jsonl --system nllb_zeroshot
!python src/eval/ifs.py --hyps experiments/results/nllb_finetuned_test_hyps.txt --refs data/processed/test.jsonl --system nllb_finetuned

## 3b. COMET (Unbabel/wmt22-comet-da)

Installed and run only now — _after_ training and the other metrics — because `unbabel-comet` pins dependencies that conflict with the training stack. Computing it last keeps training reproducible. `scripts/run_comet.py` merges the COMET score into `main_results.json` without recomputing the other metrics, then the table is regenerated.

In [ ]:
# Install COMET only now (AFTER training) so its dependency pins can't disturb the training run.
# On a Colab T4 (sm_75) there is no P100/torch-downgrade issue, so this runs cleanly on GPU.
!pip -q install unbabel-comet
!python scripts/run_comet.py --systems nllb_zeroshot,nllb_finetuned
!python src/eval/make_tables.py

In [ ]:
# Final results table (loads the canonical JSON; COMET now populated).
import json
res = json.load(open('experiments/results/main_results.json', encoding='utf-8'))
hdr = f"{'system':<40}{'chrF++':>8}{'spBLEU':>8}{'BLEU':>7}{'COMET':>8}"
print(hdr); print('-' * len(hdr))
for k in ('nllb_zeroshot', 'nllb_finetuned', 'gemini', 'gtranslate'):
    r = res.get(k)
    if r:
        print(f"{k:<40}{r.get('chrf++','-'):>8}{r.get('spbleu','-'):>8}"
              f"{r.get('bleu','-'):>7}{str(r.get('comet','-')):>8}")

## 4. Persist results back to Drive
`experiments/results/*.json`, `*_hyps.txt`, and `paper/tables/main_results.tex` are already under `PROJECT` (Drive). Also copy `/content/ckpt/best` to Drive if you want to keep the checkpoint.

**Exit criterion (Week 2):** fine-tuned chrF++ on dev beats zero-shot. If not → check data leakage / lower LR / more epochs (see plan §13).